# vlind-bench-oe — Interactive Exploration

Load a few samples from the dataset, run a model, inspect outputs, and tune the instruction prompt.

**Scoring:**
- `correct` — response matches an `expected_answer` (what the CF image shows)
- `biased`  — response matches a `biased_answer` (real-world language prior)
- `other`   — no match

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

from datasets import load_dataset
from dataset_creation.vlind_bench_oe_utils import score_response, format_prompt
import matplotlib.pyplot as plt
from IPython.display import display
import pandas as pd

## 1. Load dataset

In [ ]:
DATASET_NAME = "fabiangrob/vlind-bench-oe"
N_SAMPLES = 20  # set to None to load all

ds = load_dataset(DATASET_NAME, split="train")
if N_SAMPLES is not None:
    ds = ds.select(range(min(N_SAMPLES, len(ds))))

print(f"Loaded {len(ds)} samples")
print("Columns:", ds.column_names)
ds[0]

## 2. Browse samples

In [ ]:
def show_sample(sample, instructions=None):
    """Display one sample with its image, question, and answer sets."""
    if instructions is None:
        instructions = sample["instructions"]
    prompt = format_prompt(sample["question"], instructions)

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(sample["cf_image"])
    ax.axis("off")
    ax.set_title(f"concept: {sample['concept']}  |  id={sample['instance_id']} img={sample['cf_img_idx']}")
    plt.tight_layout()
    plt.show()

    print("PROMPT:")
    print(" ", prompt)
    print("\nEXPECTED answers:", sample["expected_answers"])
    print("BIASED  answers:",  sample["biased_answers"])
    print("true_statement: ",  sample["true_statement"])
    print("false_statement:",  sample["false_statement"])


# Browse: change the index to inspect different samples
show_sample(ds[0])

## 3. Load model

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/gemma-3-4b-it"   # change as needed
DTYPE = torch.bfloat16              # use torch.float16 for V100

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True
)
model.eval()
print("Loaded", MODEL_ID)

## 4. Tune instruction prompt

Try different instruction suffixes and see how responses change.

In [ ]:
# Candidate instructions to compare
CANDIDATE_INSTRUCTIONS = [
    "\nAnswer the question using a single word or phrase.",
    "\nAnswer in one to three words.",
    "\nGive a short answer.",
]


def run_sample(sample, instructions: str, max_new_tokens: int = 32) -> str:
    """Run model on one sample, return raw response string."""
    prompt = format_prompt(sample["question"], instructions)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["cf_image"]},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[1]
    return processor.decode(output_ids[0, input_len:], skip_special_tokens=True).strip()


# Inspect a single sample with all candidate instructions
sample = ds[0]
show_sample(sample, instructions=CANDIDATE_INSTRUCTIONS[0])

for instr in CANDIDATE_INSTRUCTIONS:
    resp = run_sample(sample, instr)
    score = score_response(resp, sample["expected_answers"], sample["biased_answers"])
    print(f"[{score:7s}]  instr={repr(instr.strip()[:40])}  →  {repr(resp)}")

## 5. Evaluate on N samples

In [ ]:
INSTRUCTIONS = "\nAnswer the question using a single word or phrase."

rows = []
for sample in ds:
    resp = run_sample(sample, INSTRUCTIONS)
    score = score_response(resp, sample["expected_answers"], sample["biased_answers"])
    rows.append({
        "instance_id":      sample["instance_id"],
        "cf_img_idx":       sample["cf_img_idx"],
        "concept":          sample["concept"],
        "question":         sample["question"],
        "expected_answers": ", ".join(sample["expected_answers"]),
        "biased_answers":   ", ".join(sample["biased_answers"]),
        "response":         resp,
        "score":            score,
    })

df = pd.DataFrame(rows)
print(df["score"].value_counts())
print(f"\nAccuracy : {(df['score']=='correct').mean():.1%}")
print(f"Bias rate: {(df['score']=='biased').mean():.1%}")
df.head(10)

## 6. Per-concept breakdown

In [ ]:
by_concept = (
    df.groupby("concept")["score"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .rename(columns={"correct": "accuracy", "biased": "bias_rate", "other": "other_rate"})
)
display(by_concept.style.format("{:.1%}").background_gradient(cmap="RdYlGn", subset=["accuracy"]))

## 7. Inspect failures

In [ ]:
# Show samples where the model was biased
biased = df[df["score"] == "biased"].reset_index(drop=True)
print(f"{len(biased)} biased responses")

for _, row in biased.head(5).iterrows():
    idx = row["instance_id"]
    sample = next(s for s in ds if s["instance_id"] == idx)
    print("\n" + "="*60)
    show_sample(sample, instructions=INSTRUCTIONS)
    print(f"Response:  {repr(row['response'])}")
    print(f"Score:     {row['score']}")